# Chapter 23 — Representation and Behavioral Diffs

**Book alignment:** Debugging AI From First Principles, Chapter 23

**Question this notebook isolates:** The upgrade to rev-B is announced with higher scores;
the pinned refund fixture drops 11/12 → 7/12. Does a byte-identical, both-revisions,
per-case diff separate **H1** (behavioral regression — harness equal, ≥3 cases flip
pass→fail) from **H2** (harness drift — the flip vanishes once hashes match) from **H3**
(redistributed competence — flips both ways)? And which rollout gate fires?

In [ ]:
import hashlib

def harness_hash(template, params):
    return hashlib.sha1(f"{template}|{sorted(params.items())}".encode()).hexdigest()[:8]

SUITE = list(range(12))          # 12 pinned refund cases

def run_case(case, *, rev, template="v3"):
    """Deterministic pass/fail for (case, rev, harness)."""
    if template != "v3":                                   # harness drift: the diff would measure this
        return (case * 7) % 3 == 0
    if rev == "A":
        return case != 4                                   # A passes 11/12 (fails case 4)
    # rev-B: breaks the 4.2-citation cases {1, 7, 9}, fixes case 4, leaves the rest unchanged
    if case in (1, 7, 9):
        return False
    if case == 4:
        return True
    return True

## 1. Attest harness equality — a drifting harness voids the diff

In [ ]:
h_a = harness_hash("v3", {"temperature": 0.0})
h_b = harness_hash("v3", {"temperature": 0.0})
print("harness hash A:", h_a, " B:", h_b)
assert h_a == h_b
# counter-example: a template bump changes the hash -> the diff would measure the harness, not the weights
assert harness_hash("v4", {"temperature": 0.0}) != h_a
print("hashes equal -> the diff can attribute to the revision")

## 2. Run both revisions byte-identical; build the per-case diff

In [ ]:
res_A = {c: run_case(c, rev="A") for c in SUITE}
res_B = {c: run_case(c, rev="B") for c in SUITE}

def label(a, b):
    return ("unchanged" if a == b else "fixed" if (b and not a) else "BROKEN")

diff = {c: label(res_A[c], res_B[c]) for c in SUITE}
counts = {k: sum(v == k for v in diff.values()) for k in ("unchanged", "fixed", "BROKEN")}
print("rev-A:", sum(res_A.values()), "/12   rev-B:", sum(res_B.values()), "/12")
print("per-case diff:", counts)
print("BROKEN cases:", [c for c, v in diff.items() if v == "BROKEN"])

assert counts["BROKEN"] >= 3                      # H1: >=3 pass->fail with harness equal
assert sum(res_A.values()) > sum(res_B.values()) # net regression wearing an upgrade announcement

## 3. Gate the rollout on the diff rows, never the headline

In [ ]:
def gate(counts):
    if counts["BROKEN"] == 0:
        return "GREEN  - roll forward, file the diff as the new baseline"
    if counts["BROKEN"] <= 3 and counts["unchanged"] >= 8:
        return "YELLOW - hold the regressed slice on rev-A, ship rev-B elsewhere, repair offline"
    return "RED    - roll back, file the diff as the incident's first artifact"

verdict = gate(counts)
print(verdict)
assert verdict.startswith("YELLOW")
# equal aggregate accuracy still hides per-example flips ('prediction churn') -> the diff is per-case
print("\nno interior signal (confidence, logit margin, attention entropy) reliably predicts which cases flip")
print("-> the full pinned suite is not optional")

## What we earned

Every model revision is a new system wearing an old name. The pinned suite (fixtures +
bundles + params + seeds) ran byte-identical against both revisions with the harness hash
attested equal; the per-case diff — 8 unchanged, 3 BROKEN, 1 fixed — convicted **H1** for
the refund slice and fired the **yellow** gate: hold the slice on rev-A, ship rev-B
elsewhere, repair offline. Equal aggregate accuracy hides per-example churn, and no cheap
signal predicts the flips, so the full suite per case is the only instrument.

**Notebook 24 / Chapter 24** opens Part V: the model is now the *author* of the work under
review, and each role it plays owes different evidence.